In [20]:
import os
from dotenv import load_dotenv
import io 
import sys
from openai import OpenAI
import gradio as gr
import subprocess
from IPython.display import display, Markdown

In [21]:
load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
grok_api_key = os.getenv("GROK_API_KEY")
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')


if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if grok_api_key:
    print(f"Grok API Key exists and begins {grok_api_key[:4]}")
else:
    print("Grok API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}")
else:
    print("OpenRouter API Key not set (and this is optional)")


OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-
Grok API Key exists and begins xai-
OpenRouter API Key exists and begins sk-or-


In [22]:
# Connect to client libraries

openai = OpenAI()

anthropic_url = "https://api.anthropic.com/v1/"
grok_url = "https://api.x.ai/v1"
groq_url = "https://api.groq.com/openai/v1"
ollama_url = "http://localhost:11434/v1"
openrouter_url = "https://openrouter.ai/api/v1"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
grok = OpenAI(api_key=grok_api_key, base_url=grok_url)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)
openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)



In [23]:
models = ["gpt-5", "claude-sonnet-4-5-20250929", "grok-4", "qwen2.5-coder", "gpt-oss:20b", "qwen/qwen3-coder-30b-a3b-instruct", "openai/gpt-oss-120b", ]

clients = {"gpt-5": openai, "claude-sonnet-4-5-20250929": anthropic, "grok-4": grok, "qwen2.5-coder": ollama, "deepseek-coder-v2": ollama, "gpt-oss:20b": ollama, "qwen/qwen3-coder-30b-a3b-instruct": openrouter}

# Want to keep costs ultra-low? Replace this with models of your choice, using the examples from yesterday

In [24]:
from system_info import retrieve_system_info, rust_toolchain_info

system_info = retrieve_system_info()
rust_info = rust_toolchain_info()

In [25]:
message = f"""
Here is a report of the system information for my computer.
I want to run a Rust compiler and C++ compiler to compile a both rust file and cpp file called main.rs, main.cpp respectively, and then execute it in the simplest way possible.
Please reply with whether I need to install a Rust toolchain or C++ Tool chain to do this. If so, please provide the simplest step by step instructions to do so.

If I'm already set up to compile both, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command for both.
Have the maximum possible runtime performance in mind; compile time can be slow. Fastest possible runtime performance for this platform is key.
Reply with the commands in markdown.

System information:
{system_info}

Rust toolchain information:
{rust_info}
"""

# response = openai.chat.completions.create(model=models[0], messages=[{"role": "user", "content": message}])
# display(Markdown(response.choices[0].message.content))

Short answer about your setup
- C++: You’re already set. Apple Clang 21.0.0 (via Xcode Command Line Tools) is installed and ready to use.
- Rust: You have rustup installed and the stable aarch64-apple-darwin toolchain is present. If rustc/cargo aren’t on PATH, you can still invoke them via rustup as shown below. Optionally add ~/.cargo/bin to PATH.

If rustc isn’t found and you want PATH-based use
- Add cargo to PATH:
  - echo 'export PATH="$HOME/.cargo/bin:$PATH"' >> ~/.zprofile
  - source ~/.zprofile
- Ensure toolchain is present: rustup update stable

Python subprocess commands (maximize runtime performance)

Rust (single-file main.rs, highest runtime performance: fat LTO, codegen-units=1, native CPU)
```python
compile_command = [
    "rustup", "run", "stable", "rustc",
    "main.rs",
    "-C", "opt-level=3",
    "-C", "lto=fat",
    "-C", "codegen-units=1",
    "-C", "target-cpu=native",
    "-C", "panic=abort",
    "-o", "main_rs"
]
run_command = ["./main_rs"]
```

C++ (main.cpp with Apple Clang, maximum runtime: Ofast + full LTO + native CPU)
```python
compile_command = [
    "clang++",
    "-std=c++20",
    "-O3", "-Ofast",
    "-flto=full",
    "-mcpu=native",
    "-DNDEBUG",
    "main.cpp",
    "-o", "main_cpp"
]
run_command = ["./main_cpp"]
```

Notes
- Ofast enables aggressive floating-point optimizations that can change numerical results. If you need strict IEEE behavior, use just -O3.
- The Rust -C panic=abort removes unwinding support; if your program or dependencies rely on unwinding, drop that flag.

In [26]:
compile_command_rust = [
    "rustc",
    "main.rs",
    "-C", "opt-level=3",
    "-C", "lto=fat",
    "-C", "codegen-units=1",
    "-C", "target-cpu=native",
    "-C", "strip=symbols",
    "-o", "main_rs",
]
run_command_rust = ["./main_rs"]

compile_command_cpp = [
    "clang++",
    "main.cpp",
    "-std=c++20",
    "-O3",
    "-flto=thin",
    "-mcpu=native",
    "-DNDEBUG",
    "-stdlib=libc++",
    "-o", "main_cpp",
]
run_command_cpp = ["./main_cpp"]



In [27]:
language = ["Rust", "C++"]

def get_extension(lang):
    if lang == "Rust":
        return "rs", compile_command_rust
    else:
        return "cpp", compile_command_cpp

In [28]:
system_prompt = f"""
Your task is to convert Python code into high performance {language[0]} or {language[1]} code.
Respond only with the target language code. Do not provide any explanation other than occasional comments.
The response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python, lang):
    extension, compile_command = get_extension(lang)
    return f"""
    Port this Python code to {lang} with the fastest possible implementation that produces identical output in the least time.
    The system information is:
    {system_info}
    Your response will be written to a file called main.{extension} and then compiled and executed; the compilation command is:
    {compile_command}
    Respond only with {lang} code.
    Python code to port:

```python
    {python}
```
    """

In [29]:
def messages_for(python, lang):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt_for(python, lang)}
    ]

In [30]:
import os

# Force to the known correct path — no ambiguity
WORK_DIR = "/Users/kavyahj/AI-Projects/AI-Code-Generator"

def write_output(code, lang):
    extension, _ = get_extension(lang)
    path = os.path.join(WORK_DIR, f"main.{extension}")
    try:
        with open(path, "w") as f:
            f.write(code)
        # Verify write actually succeeded
        if not os.path.exists(path):
            raise IOError(f"File not found after write: {path}")
        if os.path.getsize(path) == 0:
            raise IOError(f"File is empty after write: {path}")
        print(f"[write_output] OK → {path} ({os.path.getsize(path)} bytes)")
    except Exception as e:
        print(f"[write_output] FAILED: {e}")
        raise  # re-raise so compile_and_run catches it

In [31]:
def port(model, python, language):
    client = clients[model]
    reasoning_effort = "high" if 'gpt' in model else None
    response = client.chat.completions.create(model = model, messages = messages_for(python, language), reasoning_effort = reasoning_effort)
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```rust','').replace('```','')
    return reply


In [32]:
def run_python(code):
    globals_dict = {"__builtins__": __builtins__}

    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout

    return output

In [33]:
import shutil

def _find_compiler(name):
    found = shutil.which(name)
    if found:
        return found
    if name == "rustc":
        fallback = os.path.expanduser(
            "~/.rustup/toolchains/stable-aarch64-apple-darwin/bin/rustc"
        )
        if os.path.exists(fallback):
            return fallback
    return None

def compile_and_run(code, lang):
    if not code or not code.strip():
        return "Error: No code to compile. Run 'Port Code' first."
    if lang not in ["Rust", "C++"]:
        return f"Error: Invalid language '{lang}'."

    if lang == "Rust":
        source_file = os.path.join(WORK_DIR, "main.rs")
        binary      = os.path.join(WORK_DIR, "main_rs")
        rustc = _find_compiler("rustc")
        if not rustc:
            return (
                "Compiler 'rustc' not found.\n"
                "Install: curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh\n"
                "Then restart the Jupyter kernel."
            )
        compile_cmd = [
            rustc, source_file,
            "-C", "opt-level=3", "-C", "lto=fat",
            "-C", "codegen-units=1", "-C", "target-cpu=native",
            "-C", "strip=symbols", "-o", binary,
        ]
    else:
        source_file = os.path.join(WORK_DIR, "main.cpp")
        binary      = os.path.join(WORK_DIR, "main_cpp")
        compile_cmd = [
            "clang++", source_file,
            "-std=c++20", "-O3", "-flto=thin",
            "-mcpu=native", "-DNDEBUG", "-stdlib=libc++",
            "-o", binary,
        ]
    run_cmd = [binary]

    try:
        with open(source_file, "w") as f:
            f.write(code)
        if not os.path.exists(source_file) or os.path.getsize(source_file) == 0:
            raise IOError(f"File empty or missing after write: {source_file}")
    except Exception as e:
        return f"Error writing source file:\n{e}"

    try:
        subprocess.run(compile_cmd, check=True, text=True, capture_output=True, cwd=WORK_DIR)
    except subprocess.CalledProcessError as e:
        return f"Compile error:\n{e.stderr}"
    except FileNotFoundError as e:
        return f"Compiler not found: {e}"

    try:
        result = subprocess.run(run_cmd, check=True, text=True, capture_output=True, cwd=WORK_DIR)
        return result.stdout or f"(No output)\nStderr: {result.stderr}"
    except subprocess.CalledProcessError as e:
        return f"Runtime error:\n{e.stderr}"
    except FileNotFoundError:
        return f"Binary not found after compilation: {binary}"
To apply it: Select all the code in cell 13 in Jupyter, paste this in, then Shift+Enter to run just that cell. No need to restart the whole kernel.

SyntaxError: invalid syntax (2966261108.py, line 70)

[write_output] OK → /Users/kavyahj/AI-Projects/AI-Code-Generator/main.rs (1593 bytes)


In [15]:
python_hard = """# Be careful to support large numbers

def lcg(seed, a=1664525, c=1013904223, m=2**32):
    value = seed
    while True:
        value = (a * value + c) % m
        yield value
        
def max_subarray_sum(n, seed, min_val, max_val):
    lcg_gen = lcg(seed)
    random_numbers = [next(lcg_gen) % (max_val - min_val + 1) + min_val for _ in range(n)]
    max_sum = float('-inf')
    for i in range(n):
        current_sum = 0
        for j in range(i, n):
            current_sum += random_numbers[j]
            if current_sum > max_sum:
                max_sum = current_sum
    return max_sum

def total_max_subarray_sum(n, initial_seed, min_val, max_val):
    total_sum = 0
    lcg_gen = lcg(initial_seed)
    for _ in range(20):
        seed = next(lcg_gen)
        total_sum += max_subarray_sum(n, seed, min_val, max_val)
    return total_sum

# Parameters
n = 10000         # Number of random numbers
initial_seed = 42 # Initial seed for the LCG
min_val = -10     # Minimum value of random numbers
max_val = 10      # Maximum value of random numbers

# Timing the function
import time
start_time = time.time()
result = total_max_subarray_sum(n, initial_seed, min_val, max_val)
end_time = time.time()

print("Total Maximum Subarray Sum (20 runs):", result)
print("Execution Time: {:.6f} seconds".format(end_time - start_time))
"""

In [16]:


def testcase_prompt_for(code):
    return f"""
Generate 2 test cases (easy + hard) for the following Python code.
Each test must be fully self-contained with hardcoded inputs.
Each test must print the input used, the output result, and execution time.

Python code:
```python
{code}
```

Respond with only runnable Python code. No markdown fences. No explanation.
"""
def get_sp(language):
    return f"""
Your task is to convert Python code into high performance {language} code.
Respond only with {language} code. Do not provide any explanation other than occasional comments.
The {language} response needs to produce an identical output in the fastest possible time.

You are a code testing expert. Given a Python function or script, generate exactly 2 self-contained, 
runnable Python test cases:
- Test 1: Easy (small input, simple values)
- Test 2: Hard (large input, edge case, or stress test)

RULES:
1. Each test must be fully self-contained — copy the original function/code inside the test
2. Each test must hardcode its own input — NO user input(), NO stdin
3. Each test must print both the INPUT used and the OUTPUT result clearly
4. Each test must be runnable immediately with: exec() or as a standalone script
5. Wrap the two tests in a single Python code block like this:

# ── Test 1: Easy ──────────────────────────
<full self-contained code with hardcoded input>
print(f"Input: ...")
print(f"Output: result")
print(f"Time: elapsed:.4fs")

# ── Test 2: Hard ──────────────────────────
<full self-contained code with hardcoded input>
print(f"Input: ...")
print(f"Output: result")
print(f"Time: elapsed:.4fs")

Respond with ONLY the Python code. No markdown, no explanation.
"""

In [17]:
def messages_for_testcases(code, lang):
    return [
        {"role": "system", "content": get_sp(lang)},
        {"role": "user",   "content": testcase_prompt_for(code)}
    ]

def generate_testcase(model, code, lang):
    """Generate testcases AND immediately run them, return code + output."""
    client = clients[model]
    response = client.chat.completions.create(
        model=model,
        messages=messages_for_testcases(code, lang)
    )
    reply = response.choices[0].message.content
    # Strip markdown fences if model adds them
    if reply.strip().startswith("```"):
        reply = "\n".join(
            line for line in reply.splitlines()
            if not line.strip().startswith("```")
        ).strip()
    return reply

def run_testcases(testcase_code):
    """exec() the generated Python testcase code and capture its output."""
    import io, contextlib
    buf = io.StringIO()
    try:
        with contextlib.redirect_stdout(buf):
            exec(testcase_code, {"__builtins__": __builtins__})
        return buf.getvalue()
    except Exception as e:
        return f"Testcase execution error:\n{e}"

In [18]:
from styles import CSS

with gr.Blocks(css=CSS, theme=gr.themes.Monochrome(), title="Code Converter") as ui:

    # ── Row 1: three code panels ──────────────────────────────────
    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python = gr.Code(
                label="Python (Original)",
                value=python_hard,
                language="python",
                lines=26
            )
        with gr.Column(scale=6):
            cpp = gr.Code(
                label="Generated Code",
                value="",
                language="cpp",
                lines=26
            )
        with gr.Column(scale=6):
            testcases_code = gr.Code(
                label="TestCases",
                value="",
                language="python",        # ← testcases are Python, not cpp
                lines=26
            )

    # ── Row 2: action buttons ─────────────────────────────────────
    with gr.Row(elem_classes=["controlls"]):
        python_run   = gr.Button("▶ Run Python",      elem_classes=["run-btn", "py"])
        convert      = gr.Button("⇄ Port Code",       elem_classes=["side-btn", "port-btn"])
        testcase_gen = gr.Button("⚡ Gen Testcases",  elem_classes=["side-btn", "testcase-btn"])

    # ── Row 3: model/lang selectors + run buttons ─────────────────
    with gr.Row(elem_classes=["controlls"]):
        model         = gr.Dropdown(models,   value=models[0],   show_label=False)
        lang_dropdown = gr.Dropdown(language, value=language[0], show_label=False)
        cpp_run       = gr.Button("▶ Run Generated",  elem_classes=["generated-btn"])   # ← was missing
        testcase_run  = gr.Button("▶ Run TestCases",  elem_classes=["testcase-run-btn"]) # ← single definition

    # ── Row 4: output panels ──────────────────────────────────────
    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python_out   = gr.TextArea(label="Python Result",    lines=8, elem_classes=["py-out"])
        with gr.Column(scale=6):
            cpp_out      = gr.TextArea(label="Generated Result", lines=8, elem_classes=["cpp-out"])
        with gr.Column(scale=6):
            testcase_out = gr.TextArea(label="Testcases Result", lines=8, elem_classes=["cpp-out"])

    # ── Event wiring ──────────────────────────────────────────────
    python_run.click(
        fn=run_python,
        inputs=[python],
        outputs=[python_out]
    )
    convert.click(
        fn=port,
        inputs=[model, python, lang_dropdown],
        outputs=[cpp]
    )
    testcase_gen.click(
        fn=generate_testcase,
        inputs=[model, python, lang_dropdown],  # ← python source, not cpp; needs lang
        outputs=[testcases_code]
    )
    cpp_run.click(
        fn=compile_and_run,
        inputs=[cpp, lang_dropdown],            # ← needs lang to pick right compiler
        outputs=[cpp_out]
    )
    testcase_run.click(
        fn=run_testcases,
        inputs=[testcases_code],                # ← exec() the Python testcase code
        outputs=[testcase_out]
    )

ui.launch(inbrowser=True)

/var/folders/rg/66vsvnns5mdgppzcyhq7sthw0000gn/T/ipykernel_10512/1238191589.py:3: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=CSS, theme=gr.themes.Monochrome(), title="Code Converter") as ui:


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


[write_output] OK → /Users/kavyahj/AI-Projects/AI-Code-Generator/main.rs (1685 bytes)
